In [0]:
%pip install -U -qqqq mlflow databricks-langchain databricks-agents uv langgraph==0.3.4
dbutils.library.restartPython()

In [0]:
from typing import Any, Generator, Optional, Sequence, Union

import mlflow
from databricks_langchain import ChatDatabricks
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool, tool
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentChunk,
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)

############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

system_prompt = ""

############################################
# Define local tools (no UC)
############################################
@tool
def multiply_by_two(number: int) -> str:
    """Multiply an integer by two."""
    return f"{number} times two is {number * 2}"

@tool
def greet_user(name: str) -> str:
    """Return a friendly greeting to the user."""
    return f"Hello, {name}! Nice to meet you."

tools: list[BaseTool] = [multiply_by_two, greet_user]

############################################
# Define agent logic using LangGraph
############################################
def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    system_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    def should_continue(state: ChatAgentState):
        messages = state["messages"]
        last_message = messages[-1]
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    preprocessor = (
        RunnableLambda(lambda state: [{"role": "system", "content": system_prompt}] + state["messages"])
        if system_prompt else RunnableLambda(lambda state: state["messages"])
    )
    model_runnable = preprocessor | model

    def call_model(state: ChatAgentState, config: RunnableConfig):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)
    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges("agent", should_continue, {
        "continue": "tools",
        "end": END,
    })
    workflow.add_edge("tools", "agent")

    return workflow.compile()

############################################
# Agent wrapper class
############################################
class LangGraphChatAgent(ChatAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        request = {"messages": self._convert_messages_to_dict(messages)}
        messages = []
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                messages.extend(ChatAgentMessage(**msg) for msg in node_data.get("messages", []))
        return ChatAgentResponse(messages=messages)

    def predict_stream(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> Generator[ChatAgentChunk, None, None]:
        request = {"messages": self._convert_messages_to_dict(messages)}
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                yield from (
                    ChatAgentChunk(**{"delta": msg}) for msg in node_data["messages"]
                )

############################################
# Create and register the agent
############################################
mlflow.langchain.autolog()
agent = create_tool_calling_agent(llm, tools, system_prompt)
AGENT = LangGraphChatAgent(agent)
mlflow.models.set_model(AGENT)


In [0]:
from mlflow.types.agent import ChatAgentMessage

messages = [ChatAgentMessage(role="user", content="What is 6 multiplied by two?")]
response = AGENT.predict(messages=messages)
for msg in response.messages:
    print(f"{msg.role}: {msg.content}")

messages = [ChatAgentMessage(role="user", content="Say hello to Alice")]
response = AGENT.predict(messages=messages)
for msg in response.messages:
    print(f"{msg.role}: {msg.content}")

####Campaign Analyst

In [0]:
# COMMAND ----------

# MAGIC %pip install langchain==0.3.7 langchain_community==0.3.5 langgraph databricks-langchain mlflow --upgrade

# COMMAND ----------

from typing import Any, Generator, Optional, Sequence, Union
import mlflow
from databricks_langchain import ChatDatabricks
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool, tool
from langgraph.graph import END, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse, ChatAgentChunk, ChatContext

# COMMAND ----------

# Load campaign data from Spark table
campaign_df = spark.table("solutions_catalog.pih_schema.consideration_data").toPandas()

# COMMAND ----------

# Child agent tool: EngagementAgent
@tool
def analyze_engagement(campaign_id: int) -> str:
    """Analyze campaign engagement metrics for a given campaign ID."""
    campaign_data = campaign_df[campaign_df["Campaign_ID"] == campaign_id]
    if campaign_data.empty:
        return f"No data found for Campaign ID {campaign_id}."
    
    engagement = campaign_data["Engagement_Rate"].mean()
    brand_index = campaign_data["brand_index"].mean()
    digital_index = campaign_data["digital_index"].mean()

    return (
        f"Campaign ID {campaign_id} Summary:\n"
        f"- Avg Engagement Rate: {engagement:.2f}%\n"
        f"- Avg Brand Index: {brand_index:.2f}\n"
        f"- Avg Digital Index: {digital_index:.2f}"
    )

# COMMAND ----------

# Parent routing tool: CampaignAnalyst
@tool
def route_to_agent(question: str) -> str:
    """Routes the user question to the appropriate agent."""
    if "engagement" in question.lower():
        return "EngagementAgent"
    return "unknown"

# COMMAND ----------

# LLM and tool binding
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")
engagement_tools = [analyze_engagement]
campaign_tools = [route_to_agent]

# COMMAND ----------

def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    system_prompt: Optional[str] = None
) -> CompiledStateGraph:
    model = model.bind_tools(tools)

    def should_continue(state: ChatAgentState):
        last_message = state["messages"][-1]
        return "continue" if last_message.get("tool_calls") else "end"

    preprocessor = (
        RunnableLambda(lambda state: [{"role": "system", "content": system_prompt}] + state["messages"])
        if system_prompt else RunnableLambda(lambda state: state["messages"])
    )

    model_runnable = preprocessor | model

    def call_model(state: ChatAgentState, config: RunnableConfig):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)
    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))
    workflow.set_entry_point("agent")
    workflow.add_conditional_edges("agent", should_continue, {
        "continue": "tools",
        "end": END
    })
    workflow.add_edge("tools", "agent")

    return workflow.compile()

# COMMAND ----------

# Define the LangGraphChatAgent wrapper
class LangGraphChatAgent(ChatAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(self, messages: list[ChatAgentMessage], context: Optional[ChatContext] = None, custom_inputs: Optional[dict[str, Any]] = None) -> ChatAgentResponse:
        request = {"messages": self._convert_messages_to_dict(messages)}
        messages = []
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                messages.extend(ChatAgentMessage(**msg) for msg in node_data.get("messages", []))
        return ChatAgentResponse(messages=messages)

    def predict_stream(self, messages: list[ChatAgentMessage], context: Optional[ChatContext] = None, custom_inputs: Optional[dict[str, Any]] = None) -> Generator[ChatAgentChunk, None, None]:
        request = {"messages": self._convert_messages_to_dict(messages)}
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                yield from (ChatAgentChunk(**{"delta": msg}) for msg in node_data["messages"])

# COMMAND ----------

# EngagementAgent
mlflow.langchain.autolog()
engagement_graph = create_tool_calling_agent(
    llm,
    engagement_tools,
    "You are the EngagementAgent. Provide campaign engagement analysis."
)
EngagementAgent = LangGraphChatAgent(engagement_graph)

# COMMAND ----------

# CampaignAnalyst (Parent)
campaign_graph = create_tool_calling_agent(
    llm,
    campaign_tools,
    "You are the CampaignAnalyst. Evaluate user questions for ethics and delegate."
)
CampaignAnalyst = LangGraphChatAgent(campaign_graph)

# COMMAND ----------

# 🔍 Example: Route with CampaignAnalyst
messages = [ChatAgentMessage(role="user", content="What is the engagement for campaign ID 5299445?")]
response = CampaignAnalyst.predict(messages=messages)

for msg in response.messages:
    print(f"{msg.role}: {msg.content}")

# COMMAND ----------

# 🔍 Example: Analyze with EngagementAgent
messages = [ChatAgentMessage(role="user", content="Please summarize engagement for campaign ID 5299445")]
response = EngagementAgent.predict(messages=messages)

for msg in response.messages:
    print(f"{msg.role}: {msg.content}")
